# 09 · Specialist agents and handoffs

A parent agent can delegate conversation to a specialist through transfer_to_agent. Names and descriptions help the model choose the right specialist.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 3 Lite calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
orders = llm("order_specialist", "Handle order status questions. Always look up orders.",
             description="Answers order status and invoice total questions.", tools=[get_order])
catalog = llm("catalog_specialist", "Handle product stock and prices. Always look up products.",
              description="Answers product availability and pricing questions.", tools=[get_product])
router = llm("shop_router", "Delegate order questions to order_specialist and product questions to catalog_specialist.",
             sub_agents=[orders, catalog])
lab = await make_lab(router)
events = await ask(lab, "What is the status of order O1001?")
print("Participating agents:", sorted({e.author for e in events}))


## Try it

Start a fresh session and ask for stock of P103. Compare a handoff with the next notebook's AgentTool.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
